## Example 1: Dense Linear Operators

Demonstrate basic dense linear operators and matrix operations.

In [ ]:
# Example 1: Dense linear operators
from linear_operator.operators import DenseLinearOperator
import torch

# Create a positive definite matrix (A^T @ A is always positive semi-definite)
# Adding diagonal ensures positive definiteness
base_matrix = torch.randn(4, 4)
matrix = base_matrix @ base_matrix.t() + torch.eye(4) * 0.1
print(f'Original matrix shape: {matrix.shape}')

# Create dense linear operator
dense_op = DenseLinearOperator(matrix)
print(f'Dense operator shape: {dense_op.shape}')
print(f'Dense operator size: {dense_op.size()}')

# Test matrix-vector multiplication
vector = torch.randn(4)
result = dense_op.matmul(vector)
print(f'Matrix-vector product shape: {result.shape}')

# Compare with direct torch computation
direct_result = matrix @ vector
print(f'Results match: {torch.allclose(result, direct_result)}')

# Test log determinant (requires positive definite matrix)
logdet_op = torch.logdet(dense_op)
logdet_matrix = torch.logdet(matrix)
print(f'Log determinant from operator: {logdet_op:.4f}')
print(f'Log determinant from matrix: {logdet_matrix:.4f}')
print(f'Logdet matches: {torch.allclose(logdet_op, logdet_matrix)}')

# Test other supported operations
print(f'Diagonal elements: {torch.diagonal(dense_op)}')
print(f'Sum of all elements: {torch.sum(dense_op):.4f}')
print(f'Number of elements: {torch.numel(dense_op)}')

# Test matrix-matrix multiplication
other_vector = torch.randn(4)
matmul_result = dense_op @ other_vector
print(f'Matrix-vector with @ operator: {matmul_result.shape}')

# Test solve operation (linear system solving)
rhs = torch.randn(4, 1)
solution = torch.linalg.solve(dense_op, rhs)
print(f'Solve operation solution shape: {solution.shape}')

# Verify solution
verification = dense_op.matmul(solution)
print(f'Solve verification (A @ x = b): {torch.allclose(verification, rhs, atol=1e-4)}')

print('Example 1: Dense linear operators completed successfully')

## Example 2: Diagonal Linear Operators

Demonstrate efficient diagonal matrix operations with linear operators.

In [ ]:
# Example 2: Diagonal linear operators
from linear_operator.operators import DiagLinearOperator

# Create diagonal values
diag_values = torch.tensor([2.0, 3.0, 1.5, 4.0])
print(f'Diagonal values: {diag_values}')

# Create diagonal linear operator
diag_op = DiagLinearOperator(diag_values)
print(f'Diagonal operator shape: {diag_op.shape}')

# Test matrix-vector multiplication
vector = torch.randn(4)
result = diag_op.matmul(vector)
print(f'Diagonal matmul result: {result}')

# Compare with manual computation
manual_result = diag_values * vector
print(f'Manual computation: {manual_result}')
print(f'Results match: {torch.allclose(result, manual_result)}')

# Test solve operation (efficient for diagonal matrices)
# solve(rhs) computes the solution to A @ x = rhs
rhs = torch.randn(4)
try:
    solution = torch.linalg.solve(diag_op, rhs.unsqueeze(-1)).squeeze(-1)
    # Verify: diag_op @ solution should equal rhs
    verification = diag_op.matmul(solution)
    print(f'Solve operation result: {solution}')
    print(f'Solve verification (A @ x = b): {torch.allclose(verification, rhs, atol=1e-6)}')
except Exception as e:
    print(f'Solve operation: {e}')
    # Manual inverse for diagonal
    inv_diag_values = 1.0 / diag_values
    solution_manual = inv_diag_values * rhs
    print(f'Manual inverse solution: {solution_manual}')

# Test log determinant (product of diagonal elements)
logdet = torch.logdet(diag_op)
expected_logdet = torch.sum(torch.log(diag_values))
print(f'Log determinant: {logdet:.4f}, Expected: {expected_logdet:.4f}')
print(f'Log determinants match: {torch.allclose(logdet, expected_logdet)}')

# Test diagonal extraction
diag_extracted = torch.diagonal(diag_op)
print(f'Extracted diagonal: {diag_extracted}')
print(f'Diagonal matches original: {torch.allclose(diag_extracted, diag_values)}')

# Test addition with another diagonal operator
other_diag = DiagLinearOperator(torch.tensor([0.5, 0.5, 0.5, 0.5]))
sum_op = diag_op + other_diag
sum_result = sum_op.matmul(vector)
expected_sum = (diag_values + 0.5) * vector
print(f'Addition of operators: {torch.allclose(sum_result, expected_sum)}')

print('Example 2: Diagonal linear operators completed successfully')

## Example 3: Kronecker Product Operators

Demonstrate Kronecker product operations for structured matrices.

In [ ]:
# Example 3: Kronecker product operators
from linear_operator.operators import KroneckerProductLinearOperator

# Create two smaller matrices
A = torch.randn(2, 2)
B = torch.randn(3, 3)
print(f'Matrix A shape: {A.shape}')
print(f'Matrix B shape: {B.shape}')

# Create Kronecker product operator
try:
    kron_op = KroneckerProductLinearOperator(A, B)
    print(f'Kronecker product shape: {kron_op.shape}')
    
    # Test matrix-vector multiplication
    vector = torch.randn(6)  # Size should be product of dimensions
    result = kron_op.matmul(vector)
    print(f'Kronecker matmul result shape: {result.shape}')
    
    # Compare with explicit Kronecker product
    explicit_kron = torch.kron(A, B)
    explicit_result = explicit_kron @ vector
    print(f'Results match: {torch.allclose(result, explicit_result, atol=1e-6)}')
    
    print('Example 3: Kronecker product operators completed successfully')
    
except Exception as e:
    # Fallback if KroneckerProduct not available
    print(f'Kronecker product not available: {e}')
    # Demonstrate manual Kronecker product
    kron_manual = torch.kron(A, B)
    print(f'Manual Kronecker product shape: {kron_manual.shape}')
    
    vector = torch.randn(6)
    result = kron_manual @ vector
    print(f'Manual Kronecker result shape: {result.shape}')
    print('Example 3: Kronecker operations completed (fallback)')

## Example 4: Linear Operator Arithmetic

Demonstrate arithmetic operations between linear operators.

In [ ]:
# Example 4: Linear operator arithmetic
from linear_operator.operators import DenseLinearOperator, DiagLinearOperator, IdentityLinearOperator

# Create different types of operators
matrix = torch.randn(3, 3)
dense_op = DenseLinearOperator(matrix)

diag_values = torch.tensor([1.0, 2.0, 3.0])
diag_op = DiagLinearOperator(diag_values)

# Create identity operator
identity_op = IdentityLinearOperator(3)

print('Testing linear operator arithmetic...')

# Test addition
try:
    sum_op = dense_op + diag_op
    print(f'Addition result shape: {sum_op.shape}')
    
    # Test with vector
    vector = torch.randn(3)
    sum_result = sum_op.matmul(vector)
    
    # Compare with manual computation
    manual_sum = (matrix + torch.diag(diag_values)) @ vector
    print(f'Addition results match: {torch.allclose(sum_result, manual_sum, atol=1e-6)}')
except:
    print('Direct addition not supported, using manual approach')
    manual_sum = matrix + torch.diag(diag_values)
    print(f'Manual sum shape: {manual_sum.shape}')

# Test scalar multiplication
try:
    scaled_op = 2.0 * dense_op
    scaled_result = scaled_op.matmul(vector)
    manual_scaled = 2.0 * matrix @ vector
    print(f'Scaling results match: {torch.allclose(scaled_result, manual_scaled, atol=1e-6)}')
except:
    print('Scaling operation tested manually')
    raise

# Test identity operations
identity_result = identity_op.matmul(vector)
print(f'Identity preserves vector: {torch.allclose(identity_result, vector)}')
print('Example 4: Linear operator arithmetic completed successfully')

## Example 5: Batch Operations and Advanced Features

Demonstrate batch operations and advanced linear operator features.

In [ ]:
# Example 5: Batch operations and advanced features
from linear_operator.operators import CatLinearOperator, DenseLinearOperator

# Create batch of positive definite matrices
batch_size = 3
matrix_size = 4

# Generate positive definite matrices: A^T @ A + I
base_matrices = torch.randn(batch_size, matrix_size, matrix_size)
batch_matrices = torch.bmm(base_matrices, base_matrices.transpose(-2, -1))
# Add small diagonal for numerical stability and positive definiteness
batch_matrices = batch_matrices + torch.eye(matrix_size).unsqueeze(0) * 0.1

print(f'Batch matrices shape: {batch_matrices.shape}')

# Create batch linear operator
batch_op = DenseLinearOperator(batch_matrices)
print(f'Batch operator shape: {batch_op.shape}')
print(f'Batch operator batch shape: {batch_op.batch_shape}')

# Test batch matrix-vector multiplication
batch_vectors = torch.randn(batch_size, matrix_size)
batch_result = batch_op.matmul(batch_vectors.unsqueeze(-1)).squeeze(-1)
print(f'Batch result shape: {batch_result.shape}')

# Compare with manual batch computation
manual_result = torch.bmm(batch_matrices, batch_vectors.unsqueeze(-1)).squeeze(-1)
print(f'Batch results match: {torch.allclose(batch_result, manual_result, atol=1e-6)}')

# Test batch log determinants (supported operation)
batch_logdets = torch.logdet(batch_op)
print(f'Batch log determinants shape: {batch_logdets.shape}')

# Compare with manual computation
manual_logdets = torch.logdet(batch_matrices)
print(f'Batch log determinants match: {torch.allclose(batch_logdets, manual_logdets, atol=1e-6)}')

# Test batch diagonal extraction
batch_diagonals = torch.diagonal(batch_op, dim1=-2, dim2=-1)
print(f'Batch diagonals shape: {batch_diagonals.shape}')

manual_diagonals = torch.diagonal(batch_matrices, dim1=-2, dim2=-1)
print(f'Batch diagonals match: {torch.allclose(batch_diagonals, manual_diagonals)}')

# Test solve operation using torch.linalg.solve
rhs = torch.randn(batch_size, matrix_size, 1)
solutions = torch.linalg.solve(batch_op, rhs)
print(f'Batch solve solutions shape: {solutions.shape}')

# Verify solutions
verification = batch_op.matmul(solutions)
print(f'Solve verification: {torch.allclose(verification, rhs, atol=1e-4)}')

# Test batch sum operation (sum one dimension at a time)
batch_sums_rows = torch.sum(batch_op, dim=-1)  # Sum across rows
print(f'Batch row sums shape: {batch_sums_rows.shape}')

batch_sums_cols = torch.sum(batch_op, dim=-2)  # Sum across columns
print(f'Batch column sums shape: {batch_sums_cols.shape}')

# Compare with manual computation
manual_row_sums = torch.sum(batch_matrices, dim=-1)
print(f'Batch row sums match: {torch.allclose(batch_sums_rows, manual_row_sums)}')

# Test transpose operation
batch_transposed = torch.transpose(batch_op, -2, -1)
print(f'Batch transposed shape: {batch_transposed.shape}')

# Verify transpose works correctly
test_vec = torch.randn(batch_size, matrix_size, 1)
transposed_result = batch_transposed.matmul(test_vec)
manual_transposed_result = torch.bmm(batch_matrices.transpose(-2, -1), test_vec)
print(f'Transpose operation correct: {torch.allclose(transposed_result, manual_transposed_result, atol=1e-6)}')

# Test Cholesky decomposition (unique to linear_operator for structured matrices)
try:
    cholesky = torch.linalg.cholesky(batch_op)
    print(f'Cholesky decomposition shape: {cholesky.shape}')
    
    # Verify: L @ L^T = A
    cholesky_reconstruction = cholesky @ cholesky.transpose(-2, -1)
    print(f'Cholesky reconstruction matches: {torch.allclose(cholesky_reconstruction.to_dense(), batch_matrices, atol=1e-4)}')
except Exception as e:
    print(f'Cholesky decomposition: {type(e).__name__}')
    raise

# Test concatenation (unique to linear_operator)
try:
    # Create another batch operator
    extra_matrices = torch.randn(2, matrix_size, matrix_size)
    extra_matrices = torch.bmm(extra_matrices, extra_matrices.transpose(-2, -1)) + torch.eye(matrix_size) * 0.1
    extra_op = DenseLinearOperator(extra_matrices)
    
    # Concatenate along batch dimension. torch.cat does not dispatch to LinearOperators (it raises the
    # library's NotImplementedError); CatLinearOperator is linear_operator's lazy concatenation.
    concatenated = CatLinearOperator(batch_op, extra_op, dim=0)
    print(f'Concatenated operator shape: {concatenated.shape}')
    print(f'Concatenated batch shape: {concatenated.batch_shape}')
    assert concatenated.shape == torch.Size([batch_size + 2, matrix_size, matrix_size])
    assert torch.allclose(concatenated.to_dense(), torch.cat([batch_matrices, extra_matrices], dim=0))
except Exception as e:
    print(f'Concatenation: {type(e).__name__}')
    raise

print('Example 5: Batch operations and advanced features completed successfully')